In [5]:
from pyspark.sql.functions import *

StatementMeta(, 0a0c19d4-4d4a-4468-917b-e3891b3579db, 7, Finished, Available, Finished, False)

In [6]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
bronze_path = (
    "abfss://19509fa9-5985-4a20-a114-28961871f698@onelake.dfs.fabric.microsoft.com/"
    "39e67300-4542-423f-b716-14916f69a745/Files/silver_mandi_prices"
)

df = spark.read.format("delta").load(bronze_path)


StatementMeta(, 0a0c19d4-4d4a-4468-917b-e3891b3579db, 8, Finished, Available, Finished, False)

In [7]:
display(df)

StatementMeta(, 0a0c19d4-4d4a-4468-917b-e3891b3579db, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, df31104f-a394-46d0-9cd0-50f3b396410e)

In [8]:
df = df.withColumn(
    "modal_price",
    when(
        col("modal_price").isNull(),
        (col("min_price") + col("max_price")) / 2
    ).otherwise(col("modal_price"))
)

StatementMeta(, 0a0c19d4-4d4a-4468-917b-e3891b3579db, 10, Finished, Available, Finished, False)

In [9]:
df = df.filter(col("arrival_date").isNotNull())
df = df.filter(col("commodity").isNotNull())
df = df.filter(col("market").isNotNull())

StatementMeta(, 0a0c19d4-4d4a-4468-917b-e3891b3579db, 11, Finished, Available, Finished, False)

In [10]:
df = df.filter(
    (col("min_price") <= col("modal_price")) &
    (col("modal_price") <= col("max_price"))
)

StatementMeta(, 0a0c19d4-4d4a-4468-917b-e3891b3579db, 12, Finished, Available, Finished, False)

In [11]:
df = df.withColumn(
    "silver_load_timestamp",
    current_timestamp()
)

StatementMeta(, 0a0c19d4-4d4a-4468-917b-e3891b3579db, 13, Finished, Available, Finished, False)

In [12]:
display(df)

StatementMeta(, 0a0c19d4-4d4a-4468-917b-e3891b3579db, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ec9e65bb-e428-4031-bcee-edde4d6a4c4e)

In [13]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbo.silver_mandi_prices")

StatementMeta(, 0a0c19d4-4d4a-4468-917b-e3891b3579db, 15, Finished, Available, Finished, False)

In [14]:
df.show(3)
df.printSchema()

StatementMeta(, 0a0c19d4-4d4a-4468-917b-e3891b3579db, 16, Finished, Available, Finished, False)

+------------+---------------+--------------+---------+-----+---------+---------+---------+-----------+-------+-------+---------------------+
|arrival_date|      commodity|commodity_code| district|grade|   market|max_price|min_price|modal_price|  state|variety|silver_load_timestamp|
+------------+---------------+--------------+---------+-----+---------+---------+---------+-----------+-------+-------+---------------------+
|  2020-01-06|Amla(Nelli Kai)|           355|Ahmedabad|  FAQ|Ahmedabad|   2500.0|   1000.0|     1800.0|Gujarat|  Other| 2026-06-22 03:55:...|
|  2020-01-07|Amla(Nelli Kai)|           355|Ahmedabad|  FAQ|Ahmedabad|   2500.0|   1000.0|     1800.0|Gujarat|  Other| 2026-06-22 03:55:...|
|  2020-01-09|Amla(Nelli Kai)|           355|Ahmedabad|  FAQ|Ahmedabad|   2500.0|    800.0|     1700.0|Gujarat|  Other| 2026-06-22 03:55:...|
+------------+---------------+--------------+---------+-----+---------+---------+---------+-----------+-------+-------+---------------------+
only s

In [15]:
df = spark.table("silver_mandi_prices")

dim_location = (
    df.select("state", "district", "market")
    .distinct()
    .withColumn("location_key", monotonically_increasing_id())
    .select("location_key", "state", "district", "market")
)

dim_location.write.format("delta").mode("overwrite").saveAsTable("dim_location")
dim_location.show()


StatementMeta(, 0a0c19d4-4d4a-4468-917b-e3891b3579db, 17, Finished, Available, Finished, False)

+------------+-------+---------+--------------------+
|location_key|  state| district|              market|
+------------+-------+---------+--------------------+
|           0|Gujarat|Ahmedabad|               Bavla|
|           1|Gujarat|Ahmedabad|              Dholka|
|           2|Gujarat|Ahmedabad|      Dhandhuka Apmc|
|           3|Gujarat|Ahmedabad|Ahmedabad(fruit M...|
|           4|Gujarat|Ahmedabad|Ahmedabad(rajnaga...|
|           5|Gujarat|Ahmedabad|       Viramgam Apmc|
|           6|Gujarat|Ahmedabad|           Ahmedabad|
|           7|Gujarat|Ahmedabad|               Sanad|
|           8|Gujarat|Ahmedabad|Ahmedabad(chimanb...|
|           9|Gujarat|Ahmedabad|         Mandal Apmc|
|          10|Gujarat|Ahmedabad|          Sanad Apmc|
|          11|Gujarat|Ahmedabad|         Dholka Apmc|
|          12|Gujarat|Ahmedabad|Ahmedabad(chimanb...|
|          13|Gujarat|Ahmedabad|            Viramgam|
|          14|Gujarat|Ahmedabad|Ahmedabad(manekch...|
|          15|Gujarat|Ahmeda

In [16]:
dim_commodity = (
    df.select("commodity", "variety", "grade", "commodity_code")
    .distinct()
    .withColumn("commodity_key", monotonically_increasing_id())
    .select("commodity_key", "commodity", "variety", "grade", "commodity_code")
)

dim_commodity.write.format("delta").mode("overwrite").saveAsTable("dim_commodity")
dim_commodity.show()

StatementMeta(, 0a0c19d4-4d4a-4468-917b-e3891b3579db, 18, Finished, Available, Finished, False)

+-------------+--------------------+---------+-------------+--------------+
|commodity_key|           commodity|  variety|        grade|commodity_code|
+-------------+--------------------+---------+-------------+--------------+
|            0|           Moath Dal|Moath (W)|          FAQ|           258|
|            1|               Wheat| Sharbati|      Non-FAQ|             1|
|            2|              Tomato|    Other|          FAQ|            78|
|            3|      Jowar(Sorghum)|    Other|          FAQ|             5|
|            4|  Cummin Seed(Jeera)|    Other|Grade Range-3|            42|
|            5|     Chilly Capsicum|    Other|          FAQ|            88|
|            6|               Ajwan|    Ajwan|Grade Range-1|           137|
|            7|             Raddish|    Other|          FAQ|           161|
|            8|     Amla(Nelli Kai)|    Other|          FAQ|           355|
|            9|             Brinjal|    Other|          FAQ|            35|
|           

In [17]:
start_date = "2020-01-01"
end_date = "2030-12-31"

dim_date = (
    spark.sql(
        f"""
        SELECT explode(
            sequence(
                to_date('{start_date}'),
                to_date('{end_date}'),
                interval 1 day
            )
        ) AS arrival_date
        """
    )
    .withColumn("date_key", date_format(col("arrival_date"), "yyyyMMdd").cast("int"))
    .withColumn("year", year(col("arrival_date")))
    .withColumn("month", month(col("arrival_date")))
    .withColumn("day", dayofmonth(col("arrival_date")))
    .withColumn("month_name", date_format(col("arrival_date"), "MMMM"))
    .withColumn("weekday", date_format(col("arrival_date"), "EEEE"))
    .withColumn("quarter", expr("quarter(arrival_date)"))
)

dim_date.write.format("delta").mode("overwrite").saveAsTable("dim_date")
dim_date.show()


StatementMeta(, 0a0c19d4-4d4a-4468-917b-e3891b3579db, 19, Finished, Available, Finished, False)

+------------+--------+----+-----+---+----------+---------+-------+
|arrival_date|date_key|year|month|day|month_name|  weekday|quarter|
+------------+--------+----+-----+---+----------+---------+-------+
|  2020-01-01|20200101|2020|    1|  1|   January|Wednesday|      1|
|  2020-01-02|20200102|2020|    1|  2|   January| Thursday|      1|
|  2020-01-03|20200103|2020|    1|  3|   January|   Friday|      1|
|  2020-01-04|20200104|2020|    1|  4|   January| Saturday|      1|
|  2020-01-05|20200105|2020|    1|  5|   January|   Sunday|      1|
|  2020-01-06|20200106|2020|    1|  6|   January|   Monday|      1|
|  2020-01-07|20200107|2020|    1|  7|   January|  Tuesday|      1|
|  2020-01-08|20200108|2020|    1|  8|   January|Wednesday|      1|
|  2020-01-09|20200109|2020|    1|  9|   January| Thursday|      1|
|  2020-01-10|20200110|2020|    1| 10|   January|   Friday|      1|
|  2020-01-11|20200111|2020|    1| 11|   January| Saturday|      1|
|  2020-01-12|20200112|2020|    1| 12|   January

In [18]:
fact_mandi_prices = (
    df.join(dim_location, on=["state", "district", "market"], how="left")
    .join(dim_commodity, on=["commodity", "variety", "grade", "commodity_code"], how="left")
    .join(dim_date, on=["arrival_date"], how="left")
    .select(
        "location_key",
        "commodity_key",
        "date_key",
        "min_price",
        "max_price",
        "modal_price"
    )
)

fact_mandi_prices.write.format("delta").mode("overwrite").saveAsTable("fact_mandi_prices")
fact_mandi_prices.show(5)

StatementMeta(, 0a0c19d4-4d4a-4468-917b-e3891b3579db, 20, Finished, Available, Finished, False)

+------------+-------------+--------+---------+---------+-----------+
|location_key|commodity_key|date_key|min_price|max_price|modal_price|
+------------+-------------+--------+---------+---------+-----------+
|           6|            8|20200106|   1000.0|   2500.0|     1800.0|
|           6|            8|20200107|   1000.0|   2500.0|     1800.0|
|           6|            8|20200109|    800.0|   2500.0|     1700.0|
|           6|            8|20200110|    700.0|   2000.0|     1450.0|
|           6|            8|20200111|    700.0|   2000.0|     1400.0|
+------------+-------------+--------+---------+---------+-----------+
only showing top 5 rows



In [19]:
print("Original silver rows:", df.count())
print("Fact table rows:", fact_mandi_prices.count())
print("Nulls in location_key:", fact_mandi_prices.filter(col("location_key").isNull()).count())
print("Nulls in commodity_key:", fact_mandi_prices.filter(col("commodity_key").isNull()).count())
print("Nulls in date_key:", fact_mandi_prices.filter(col("date_key").isNull()).count())

StatementMeta(, 0a0c19d4-4d4a-4468-917b-e3891b3579db, 21, Finished, Available, Finished, False)

Original silver rows: 58532
Fact table rows: 58532
Nulls in location_key: 0
Nulls in commodity_key: 0
Nulls in date_key: 0
